# Practice: Explain the Code — Valuing a Private Loan

## Background
 The fastest-growing corner of modern finance — **private credit**, loans negotiated directly between a fund and a company — rarely writes terms that clean. A realistic term sheet line reads:

> *"€100M unitranche, 5-year bullet, **6% cash + 4% PIK**, issued at **97 (3 points OID)**."*

Two instruments of the trade hide in that sentence:

- **OID — original issue discount.** The lender hands over less than face value (here 97 for a face of 100) but interest and repayment run on the full face. Economically it is yield dressed up as a discount — and often a fee dressed up as yield.
- **PIK — payment-in-kind.** Instead of paying interest in cash, the borrower pays it *in more debt*: the interest is added to the balance, which then itself earns interest. The lender books income every year; the cash arrives — if it arrives — at the very end.

Why do these exist? OID lets arrangers take economics upfront without raising the headline coupon; PIK lets a cash-strapped borrower (a freshly leveraged LBO company, say) conserve cash in the lean years. Both are everywhere in direct lending — and both drive a wedge between the **stated coupon**, the **promised yield**, and the **realized return**. Today's forty lines of code: a loan that pays 8% cash can yield over 10%, and a loan that accrues 10% on paper can realize exactly 0%.


## Problem

A credit fund has lent money to a company under the following contract:

- The loan's **face value is €100M** — but the fund actually **advanced only €97M**.
- Each year, the borrower pays **6% of the current loan balance in cash**.
- Another **4% per year is not paid in cash** — it is **added to the balance**, which therefore grows over time (and next year's interest runs on the bigger balance).
- After **5 years**, the borrower repays the entire (grown) balance in one payment.

*(In industry jargon this is a "5-year unitranche, 6% cash + 4% PIK, issued at 97.")*

Question: *what rate of return does this contract promise*, and *what is the loan worth today*, given where market interest rates are right now?

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import fsolve
import matplotlib.pyplot as plt

In [2]:
# GS5: Market Yield on U.S. Treasury Securities at 5-Year Constant Maturity
url = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=GS5'
rates = pd.read_csv(url, index_col=0, parse_dates=True).dropna()

In [3]:
r_f = rates.iloc[-1, 0] / 100
credit_spread = 0.045
r_req = r_f + credit_spread

In [4]:
principal = 100
OID = 0.03
cash_rate = 0.06
pik_rate = 0.04
maturity = 5

In [5]:
t = np.arange(1, maturity+1)
loan_balances = principal * (1+pik_rate)**(t - 1)

In [6]:
flows = np.zeros(maturity+1)
flows[0] = -principal*(1-OID)
flows[1:] = cash_rate * loan_balances
flows[-1] += principal * (1+pik_rate)**maturity

In [7]:
times = np.arange(len(flows))
y = fsolve(lambda r: np.sum(flows / (1 + r)**times), 0.10)[0]

In [8]:
value = np.sum(flows[1:] / (1 + r_req)**times[1:])

In [9]:
print(f"5-year Treasury yield today: {r_f:.2%}  ->  required yield: {r_req:.2%}")
print(f"Promised yield of the loan:  {y:.2%}")
print(f"Value of the loan today:     {value:.2f}   (the fund paid 97.00)")

5-year Treasury yield today: 4.93%  ->  required yield: 9.43%
Promised yield of the loan:  10.75%
Value of the loan today:     102.36   (the fund paid 97.00)
